### Create a classificator model based on LLM-generated features and TF-IDF terms for the hate speech dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from utils.utils import preprocessing, process_txt_files
import shutup
shutup.please()

In [2]:
folder_path = "../../data/outputs/hazard"
df_llm_features = process_txt_files(folder_path, "hazard")
df_llm_features = df_llm_features.drop(columns=["id", "custom_id", 'recall_date', 'product_batch_code'])
# Convert any list values in the DataFrame to strings
for col in df_llm_features.columns:
    df_llm_features[col] = df_llm_features[col].apply(lambda x: str(x) if isinstance(x, list) else x)

# Now apply get_dummies
df_llm_features = pd.get_dummies(df_llm_features, sparse=False, prefix_sep='_')

In [3]:
import requests
import pandas as pd
from io import StringIO

# URLs for the files
urls = [
    "https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_train.csv",
    "https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_valid.csv",
    "https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_test.csv"
]

# Load each file into a DataFrame
dataframes = []
for url in urls:
    response = requests.get(url)
    response.raise_for_status()  # Raise an error for bad status codes
    csv_data = StringIO(response.text)  # Convert text to a file-like object
    df_orig = pd.read_csv(
        csv_data,
        engine='python',             # supports multiline quoted fields
        quoting=0,                   # QUOTE_MINIMAL
        quotechar='"',
        on_bad_lines='warn'          # skip lines that still cause issues
    )
    dataframes.append(df_orig)

# Access the DataFrames
train_df, valid_df, test_df = dataframes
train_df = pd.concat([train_df, valid_df])

train_df['text'] = train_df['text'].apply(lambda x: preprocessing(x))
train_df['title'] = train_df['title'].apply(lambda x: preprocessing(x))
train_df['combined'] = train_df['title'] + ' ' +  train_df['text']


test_df['text'] = test_df['text'].apply(lambda x: preprocessing(x))
test_df['title'] = test_df['title'].apply(lambda x: preprocessing(x))
test_df['combined'] = test_df['title'] + ' ' +  test_df['text']
# Example: Display the first few rows of the training DataFrame

In [4]:
X_train_llm_features = df_llm_features.iloc[:train_df.shape[0]]
X_test_llm_features = df_llm_features.tail(len(test_df))
y_train = train_df['hazard-category']
hazard_true = test_df['hazard-category']
# y_valid = valid_df['hazard-category']

In [5]:
X_train_text = train_df['combined']
X_test_text = test_df['combined']

In [6]:
def tf_idf(train, test):
    vectorizer = TfidfVectorizer()
    train_tfidf = vectorizer.fit_transform(train)
    test_tfdidf = vectorizer.transform(test)
    return pd.DataFrame(train_tfidf.toarray(), columns=vectorizer.get_feature_names_out()),  pd.DataFrame(test_tfdidf.toarray(), columns=vectorizer.get_feature_names_out())

In [7]:
X_train_text, X_test_text = tf_idf(X_train_text, X_test_text)

In [8]:
# Merge the LLM features with the tf-idf features
X_train = pd.concat((X_train_text.reset_index(drop=True), X_train_llm_features.reset_index(drop=True)), axis=1)
X_test = pd.concat((X_test_text.reset_index(drop=True), X_test_llm_features.reset_index(drop=True)), axis=1)

In [9]:
# 1) Libraries
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score


# ----------------------------------------------------------------
# 4) Definice modelu RandomForestClassifier
model = RandomForestClassifier(random_state=42)

# 5) Nastavení rozsahu parametrů pro RandomizedSearchCV
param_dist = {
    "n_estimators": [50, 100, 200],       # Počet stromů v lese
    "max_depth": [3, 5, 10, None],        # Maximální hloubka stromu
    "min_samples_split": [2, 5, 10],      # Minimální počet vzorků pro split
    "min_samples_leaf": [1, 2, 5],        # Minimální počet vzorků v listu
}

# 6) Konfigurace RandomizedSearchCV (n_iter a cv lze upravit dle potřeby)
random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=10,             # kolik náhodných kombinací parametrů prozkoumat
    cv=5,                  # 5-fold cross-validace
    scoring="f1_macro",    # metrika, dle které se bude model porovnávat
    random_state=42,
    n_jobs=-1,             # využití všech CPU jader pro rychlejší výpočet
    verbose=1
)

# 7) Trénink modelu s vyhledáváním nejlepších hyperparametrů
random_search.fit(X_train, y_train)

# 8) Vypsání nejlepších parametrů a skóre
print("Nejlepší parametry:", random_search.best_params_)
print("Nejlepší skóre na trénovací cross-validaci:", random_search.best_score_)

# 9) Ověření na testovací sadě
best_model = random_search.best_estimator_  # získáme nejlepší nalezený model
hazard_pred = best_model.predict(X_test)

# 10) Vyhodnocení
print("Přesnost na testu:", accuracy_score(hazard_true, hazard_pred))
print("Classification report na testu:")
print(classification_report(hazard_true ,hazard_pred, zero_division=0))


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Nejlepší parametry: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None}
Nejlepší skóre na trénovací cross-validaci: 0.5577998224035279
Přesnost na testu: 0.9297893681043129
Classification report na testu:
                                precision    recall  f1-score   support

                     allergens       0.94      0.99      0.97       365
                    biological       0.95      0.99      0.97       343
                      chemical       0.94      0.87      0.90        52
food additives and flavourings       1.00      0.50      0.67         4
                foreign bodies       0.89      0.99      0.94       111
                         fraud       0.88      0.69      0.78        75
                     migration       0.00      0.00      0.00         1
          organoleptic aspects       1.00      0.20      0.33        10
                  other hazard       0.75      0.